# Plant Health Diagnostics: Training on Google Colab
Trains EfficientNet-B0 and produces `plant_health_effnet_b0.pt`, which you download and place in your VS Code project's `models/` folder.

**Before you start:** Runtime > Change runtime type > **T4 GPU**, then run the cells top to bottom.

In [ ]:
import torch, os
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only (enable T4 GPU!)")
os.makedirs("/content/plant_health/models", exist_ok=True)
%cd /content/plant_health

## 1. Write the project code files

In [ ]:
%%writefile config.py
"""Shared configuration for the Plant Health Diagnostic project."""
from pathlib import Path

ROOT = Path(__file__).resolve().parent
DATA_DIR = ROOT / "data"                       # data/<class_name>/*.jpg
MODEL_DIR = ROOT / "models"
CHECKPOINT = MODEL_DIR / "plant_health_effnet_b0.pt"

IMG_SIZE = 224
# ImageNet statistics (required because the backbone was pre-trained on ImageNet)
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

# Below this top-1 probability the app flags the result as "low confidence".
LOW_CONFIDENCE = 0.50
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

In [ ]:
%%writefile dataset.py
"""Dataset + DataLoader setup.

Expected folder layout (one sub-folder per class; folder name = class label):

    data/
      healthy/              *.jpg
      nitrogen_deficiency/  *.jpg
      phosphorus_deficiency/*.jpg
      potassium_deficiency/ *.jpg
      dead_necrotic/        *.jpg

Any additional folder (e.g. magnesium_deficiency/) is picked up automatically.
"""
import random
from collections import defaultdict

import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

from config import IMG_SIZE, MEAN, STD


def get_transforms(train: bool) -> transforms.Compose:
    if train:
        return transforms.Compose([
            transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),          # leaves have no canonical "up"
            transforms.RandomRotation(15),
            # Keep hue shifts tiny: yellowing/purpling IS the diagnostic signal,
            # so aggressive hue jitter would corrupt the labels.
            transforms.ColorJitter(brightness=0.25, contrast=0.25,
                                   saturation=0.2, hue=0.02),
            transforms.ToTensor(),
            transforms.Normalize(MEAN, STD),
        ])
    return transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])


def stratified_split(targets, val_fraction, seed):
    """Per-class split so every class appears in both train and val."""
    by_class = defaultdict(list)
    for idx, t in enumerate(targets):
        by_class[t].append(idx)
    rng = random.Random(seed)
    train_idx, val_idx = [], []
    for idxs in by_class.values():
        rng.shuffle(idxs)
        n_val = max(1, int(len(idxs) * val_fraction))
        val_idx += idxs[:n_val]
        train_idx += idxs[n_val:]
    return train_idx, val_idx


def build_dataloaders(data_dir, batch_size=32, val_fraction=0.2,
                      num_workers=2, seed=42):
    # Two ImageFolder views of the same files: one with train aug, one without.
    train_full = datasets.ImageFolder(data_dir, transform=get_transforms(True))
    val_full = datasets.ImageFolder(data_dir, transform=get_transforms(False))

    train_idx, val_idx = stratified_split(train_full.targets, val_fraction, seed)
    train_ds, val_ds = Subset(train_full, train_idx), Subset(val_full, val_idx)

    # Inverse-frequency class weights to counter imbalance
    # (dead plants / rare deficiencies are usually under-represented).
    counts = torch.zeros(len(train_full.classes))
    for i in train_idx:
        counts[train_full.targets[i]] += 1
    class_weights = counts.sum() / (len(counts) * counts.clamp(min=1))

    pin = torch.cuda.is_available()
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=pin)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                            num_workers=num_workers, pin_memory=pin)
    return train_loader, val_loader, train_full.classes, class_weights

In [ ]:
%%writefile model.py
"""Model definition: EfficientNet-B0 backbone with a replaced classification head."""
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights


def build_model(num_classes: int, pretrained: bool = True,
                freeze_backbone: bool = True, unfreeze_last_n: int = 2,
                dropout: float = 0.3) -> nn.Module:
    """
    Args:
        freeze_backbone: freeze the convolutional feature extractor.
        unfreeze_last_n: keep the last N feature blocks trainable (they hold the
            most task-specific, high-level features). 0 = pure linear probe.
    """
    weights = EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
    model = efficientnet_b0(weights=weights)

    if freeze_backbone:
        for p in model.features.parameters():
            p.requires_grad = False
        if unfreeze_last_n > 0:
            for block in list(model.features)[-unfreeze_last_n:]:
                for p in block.parameters():
                    p.requires_grad = True

    in_features = model.classifier[1].in_features          # 1280 for B0
    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout),
        nn.Linear(in_features, num_classes),               # new head (trainable)
    )
    return model


def param_groups(model: nn.Module, head_lr: float, backbone_lr: float):
    """Higher LR for the fresh head, lower LR for the unfrozen backbone blocks."""
    head = [p for p in model.classifier.parameters() if p.requires_grad]
    backbone = [p for p in model.features.parameters() if p.requires_grad]
    groups = [{"params": head, "lr": head_lr}]
    if backbone:
        groups.append({"params": backbone, "lr": backbone_lr})
    return groups


# --- ResNet-50 alternative -------------------------------------------------
# from torchvision.models import resnet50, ResNet50_Weights
# m = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
# for n, p in m.named_parameters():
#     p.requires_grad = n.startswith(("layer4", "fc"))
# m.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.fc.in_features, num_classes))
# (then use m.fc / m.layer4 in param_groups instead of classifier / features)

In [ ]:
%%writefile train.py
"""Fine-tune EfficientNet-B0 on the plant-health dataset.

Usage:
    python train.py --data-dir data --epochs 15
"""
import argparse
import json
import time

import torch
import torch.nn as nn

from config import DATA_DIR, CHECKPOINT, MODEL_DIR
from dataset import build_dataloaders
from model import build_model, param_groups


def run_epoch(model, loader, criterion, device, optimizer=None, scaler=None, use_amp=False):
    """One pass over `loader`. Trains if an optimizer is given, otherwise evaluates."""
    training = optimizer is not None
    model.train(training)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                logits = model(images)
                loss = criterion(logits, labels)
            if training:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item() * images.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            n += images.size(0)
    return total_loss / n, correct / n


@torch.no_grad()
def per_class_report(model, loader, class_names, device):
    model.eval()
    k = len(class_names)
    cm = torch.zeros(k, k, dtype=torch.long)           # rows=true, cols=predicted
    for images, labels in loader:
        preds = model(images.to(device)).argmax(1).cpu()
        for t, p in zip(labels, preds):
            cm[t, p] += 1
    print("\nPer-class validation accuracy:")
    for i, name in enumerate(class_names):
        total = cm[i].sum().item()
        acc = cm[i, i].item() / total if total else float("nan")
        print(f"  {name:<28} {acc:6.1%}  ({total} images)")
    print("\nConfusion matrix (rows=true, cols=pred):")
    print("  classes:", class_names)
    print(cm)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-dir", default=str(DATA_DIR))
    ap.add_argument("--epochs", type=int, default=15)
    ap.add_argument("--batch-size", type=int, default=32)
    ap.add_argument("--head-lr", type=float, default=1e-3)
    ap.add_argument("--backbone-lr", type=float, default=1e-4)
    ap.add_argument("--weight-decay", type=float, default=1e-2)
    ap.add_argument("--unfreeze-last-n", type=int, default=2)
    ap.add_argument("--val-fraction", type=float, default=0.2)
    ap.add_argument("--num-workers", type=int, default=2)   # use 0 on Windows if you hit errors
    ap.add_argument("--seed", type=int, default=42)
    args = ap.parse_args()

    torch.manual_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = device.type == "cuda"
    print(f"Device: {device} | AMP: {use_amp}")

    train_loader, val_loader, class_names, class_weights = build_dataloaders(
        args.data_dir, args.batch_size, args.val_fraction, args.num_workers, args.seed)
    print(f"Classes ({len(class_names)}): {class_names}")
    print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

    model = build_model(len(class_names), unfreeze_last_n=args.unfreeze_last_n).to(device)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Trainable params: {trainable:,} / {total:,}")

    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device), label_smoothing=0.05)
    optimizer = torch.optim.AdamW(
        param_groups(model, args.head_lr, args.backbone_lr), weight_decay=args.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    best_acc, history = 0.0, []
    for epoch in range(1, args.epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = run_epoch(model, train_loader, criterion, device,
                                    optimizer, scaler, use_amp)
        va_loss, va_acc = run_epoch(model, val_loader, criterion, device, use_amp=use_amp)
        scheduler.step()
        history.append(dict(epoch=epoch, train_loss=tr_loss, train_acc=tr_acc,
                            val_loss=va_loss, val_acc=va_acc))
        flag = ""
        if va_acc > best_acc:
            best_acc = va_acc
            torch.save({"state_dict": model.state_dict(),
                        "class_names": class_names,
                        "val_acc": va_acc,
                        "args": vars(args)}, CHECKPOINT)
            flag = "  <- saved best"
        print(f"Epoch {epoch:02d}/{args.epochs} | train {tr_loss:.4f}/{tr_acc:.1%} | "
              f"val {va_loss:.4f}/{va_acc:.1%} | {time.time() - t0:.0f}s{flag}")

    (MODEL_DIR / "history.json").write_text(json.dumps(history, indent=2))

    # Final report using the best checkpoint
    model.load_state_dict(torch.load(CHECKPOINT, map_location=device)["state_dict"])
    per_class_report(model, val_loader, class_names, device)
    print(f"\nBest val accuracy: {best_acc:.1%} | checkpoint: {CHECKPOINT}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile knowledge_base.py
"""Rule-based mapping: model class -> agronomy findings and action plan.

Keys must match your dataset folder names (lower-case, underscores).
To add a class (e.g. magnesium_deficiency), train with that folder present and
add one more entry below; unknown classes fall back to GENERIC.

NOTE: product rates are intentionally NOT hard-coded. Always follow the label
of the specific fertilizer and confirm with a soil/tissue test.
"""

KNOWLEDGE_BASE = {
    "healthy": {
        "title": "Healthy plant",
        "category": "healthy",
        "severity": "none",
        "summary": "No visible stress pattern detected. Foliage colour and structure look normal.",
        "symptoms": ["Uniform green foliage", "Turgid leaves", "No lesions or scorching"],
        "root_causes": [],
        "soil_amendments": ["No amendment needed. Maintain current feeding programme."],
        "precautions": [
            "Keep scouting weekly; early symptoms start on older or newest leaves.",
            "Photograph the plant on a regular schedule to track change.",
        ],
        "prevention": [
            "Soil-test once or twice a year and keep pH in the crop's preferred range.",
            "Water at the base, mulch to hold moisture, rotate crops each season.",
        ],
    },
    "nitrogen_deficiency": {
        "title": "Nitrogen (N) deficiency",
        "category": "nutrient",
        "severity": "moderate",
        "summary": "Pattern consistent with nitrogen shortage: general yellowing that starts on "
                   "the OLDER, lower leaves because N is mobile and moves to new growth.",
        "symptoms": ["Uniform pale green to yellow older leaves (chlorosis)",
                     "Stunted, spindly growth", "Reduced leaf size and early leaf drop"],
        "root_causes": ["Low soil organic matter or leached soil (heavy rain / over-watering)",
                        "Heavy feeders in nutrient-depleted beds",
                        "High-carbon mulch or wood chips tying up N (immobilisation)"],
        "soil_amendments": [
            "Quick fix: water-soluble N or a diluted liquid feed (fish emulsion, seaweed blend) per label.",
            "Slow release: well-composted manure, blood meal, or a balanced slow-release fertilizer.",
            "Long term: add compost and grow legume cover crops / green manure.",
        ],
        "precautions": [
            "Do not over-apply: excess N gives soft growth, attracts pests and delays fruiting.",
            "Split applications into small doses rather than one heavy dose.",
            "Rule out root problems or waterlogging, which mimic N deficiency.",
        ],
        "prevention": ["Annual compost top-dressing", "Avoid over-irrigation and nutrient leaching",
                       "Rotate heavy feeders with legumes"],
    },
    "phosphorus_deficiency": {
        "title": "Phosphorus (P) deficiency",
        "category": "nutrient",
        "severity": "moderate",
        "summary": "Pattern consistent with phosphorus shortage: dark green or purplish/bronze "
                   "older leaves and slow, stunted growth, often in cold or acidic/alkaline soils.",
        "symptoms": ["Dark green, purple or reddish tint on older leaves and stems",
                     "Slow growth and delayed flowering", "Poor root development"],
        "root_causes": ["Cold soil (P uptake drops sharply at low temperature)",
                        "Soil pH too low (<5.5) or too high (>7.5), which locks P up",
                        "Low available P in soil; compacted or waterlogged roots"],
        "soil_amendments": [
            "Bone meal or composted manure for organic P; superphosphate per label.",
            "Rock phosphate works slowly and only in acidic soil.",
            "Correct pH first (lime if acidic, sulfur if alkaline) so existing P becomes available.",
        ],
        "precautions": [
            "Check soil temperature; purpling in cold spells may resolve by itself.",
            "Avoid excess P: it can induce zinc and iron deficiency and runs off into waterways.",
            "Apply near the root zone; P barely moves in soil.",
        ],
        "prevention": ["Maintain pH 6.0-7.0 for most crops", "Use mycorrhizal inoculants at planting",
                       "Warm the soil early in season (mulch film, raised beds)"],
    },
    "potassium_deficiency": {
        "title": "Potassium (K) deficiency",
        "category": "nutrient",
        "severity": "moderate to high",
        "summary": "Pattern consistent with potassium shortage: yellowing and scorched, brown "
                   "margins and tips on older leaves, weak stems and poor fruit quality.",
        "symptoms": ["Yellow then brown leaf edges and tips (marginal scorch) on older leaves",
                     "Weak stems and lodging", "Poor fruit set, uneven ripening"],
        "root_causes": ["Sandy soils with low K-holding capacity", "Excess calcium or magnesium competing with K",
                        "Drought stress or heavy fruiting load"],
        "soil_amendments": [
            "Potassium sulfate (sulfate of potash) per label; use sulfate form for chloride-sensitive crops.",
            "Organic options: compost, kelp meal, greensand (slow).",
            "Wood ash supplies K but raises pH sharply; use sparingly and only on acidic soil.",
        ],
        "precautions": [
            "Over-application of K can induce magnesium and calcium deficiency.",
            "Confirm with a tissue test; salt injury and drought produce similar scorch.",
            "Keep soil evenly moist so K can be taken up.",
        ],
        "prevention": ["Regular soil tests, particularly on sandy soils", "Mulch to conserve moisture",
                       "Balanced N-P-K feeding, especially during fruiting"],
    },
    "dead_necrotic": {
        "title": "Dead / necrotic specimen (post-mortem)",
        "category": "dead",
        "severity": "critical",
        "summary": "Tissue appears largely dead (brown/black, desiccated or collapsed). A photo cannot "
                   "by itself prove WHY the plant died, so use the post-mortem checklist to find the cause.",
        "symptoms": ["Widespread brown/black or crispy tissue", "Collapsed stems", "No green tissue"],
        "root_causes": [
            "Root rot / waterlogging (mushy, dark, foul-smelling roots)",
            "Prolonged drought or heat scorch (dry, brittle roots and crispy leaves)",
            "Frost or freeze damage (sudden blackening, water-soaked tissue)",
            "Vascular wilt, stem canker or other disease (brown streaking inside stem)",
            "Pests (borers, root grubs, nematodes) or herbicide drift / fertilizer burn",
            "Severe long-term nutrient deficiency or toxicity",
        ],
        "soil_amendments": [
            "Do not re-fertilise a dead plant; test the soil (pH, EC/salinity, N-P-K) before replanting.",
            "If root rot or disease is suspected: improve drainage, add compost, or solarise the soil.",
            "If salts/fertilizer burn is suspected: flush the bed with plenty of water.",
        ],
        "precautions": [
            "Scratch test: scrape a stem; green tissue underneath means the plant may still recover. "
            "Cut back dead growth and keep watering sensibly.",
            "Inspect roots: firm and white = healthy; brown, slimy and smelly = rot.",
            "Remove diseased plants and dispose of them (bin, not compost).",
            "Disinfect tools and pots (10% bleach or 70% alcohol).",
            "Isolate neighbouring plants and check them for the same symptoms.",
        ],
        "prevention": ["Match watering to soil drying, not the calendar", "Use well-drained soil and clean pots",
                       "Protect from frost; harden off seedlings", "Rotate crops; do not replant the same family in diseased soil"],
    },
}

GENERIC = {
    "title": "Unrecognised class",
    "category": "unknown",
    "severity": "unknown",
    "summary": "No specific guidance is stored for this class. Add an entry to knowledge_base.py.",
    "symptoms": [], "root_causes": [], "soil_amendments": [],
    "precautions": ["Confirm with a soil test and an agronomist."], "prevention": [],
}


def normalise(label: str) -> str:
    return label.strip().lower().replace(" ", "_").replace("-", "_")


def get_recommendations(label: str) -> dict:
    return KNOWLEDGE_BASE.get(normalise(label), GENERIC)

In [ ]:
%%writefile inference.py
"""Inference + diagnostic engine.

CLI:   python inference.py path/to/plant.jpg
Code:  diagnoser = PlantDiagnoser(); result = diagnoser.predict("plant.jpg")
"""
import io
import sys
from pathlib import Path
from typing import Union

import torch
from PIL import Image

from config import CHECKPOINT, LOW_CONFIDENCE
from dataset import get_transforms
from knowledge_base import get_recommendations
from model import build_model

ImageInput = Union[str, Path, bytes, Image.Image]


class PlantDiagnoser:
    def __init__(self, checkpoint: Union[str, Path] = CHECKPOINT, device: str = None):
        checkpoint = Path(checkpoint)
        if not checkpoint.exists():
            raise FileNotFoundError(
                f"Checkpoint not found at {checkpoint}. Run `python train.py` first.")
        self.device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
        ckpt = torch.load(checkpoint, map_location=self.device)
        self.class_names = ckpt["class_names"]
        self.val_acc = ckpt.get("val_acc")
        self.model = build_model(len(self.class_names), pretrained=False)  # weights come from ckpt
        self.model.load_state_dict(ckpt["state_dict"])
        self.model.to(self.device).eval()
        self.transform = get_transforms(train=False)

    @staticmethod
    def _load_image(image: ImageInput) -> Image.Image:
        if isinstance(image, Image.Image):
            return image.convert("RGB")
        if isinstance(image, (bytes, bytearray)):
            return Image.open(io.BytesIO(image)).convert("RGB")
        return Image.open(image).convert("RGB")

    @torch.no_grad()
    def predict(self, image: ImageInput, top_k: int = 3) -> dict:
        img = self._load_image(image)
        x = self.transform(img).unsqueeze(0).to(self.device)
        # Light test-time augmentation: average original + horizontal flip
        logits = (self.model(x) + self.model(torch.flip(x, dims=[3]))) / 2
        probs = torch.softmax(logits, dim=1).squeeze(0).cpu()

        k = min(top_k, len(self.class_names))
        top_p, top_i = probs.topk(k)
        top = [{"label": self.class_names[i], "confidence": float(p)}
               for p, i in zip(top_p, top_i)]
        all_probs = {name: float(p) for name, p in zip(self.class_names, probs)}

        primary = top[0]
        margin = top[0]["confidence"] - (top[1]["confidence"] if k > 1 else 0.0)
        warnings = []
        if primary["confidence"] < LOW_CONFIDENCE:
            warnings.append("Low confidence: retake the photo (good light, plain background, "
                            "affected leaf in focus) and confirm with a soil or tissue test.")
        elif margin < 0.15:
            warnings.append(f"Close call between '{top[0]['label']}' and '{top[1]['label']}'. "
                            "Treat the diagnosis as provisional.")

        return {
            "primary_label": primary["label"],
            "primary_confidence": primary["confidence"],
            "top_k": top,
            "all_probabilities": all_probs,
            "warnings": warnings,
            "recommendations": get_recommendations(primary["label"]),
        }


def print_report(result: dict) -> None:
    rec = result["recommendations"]
    print(f"\nDiagnosis: {rec['title']}  ({result['primary_confidence']:.1%} confidence)")
    print(f"Severity : {rec['severity']}\n{rec['summary']}\n")
    for w in result["warnings"]:
        print(f"! {w}")
    print("\nTop predictions:")
    for t in result["top_k"]:
        print(f"  {t['label']:<28} {t['confidence']:.1%}")
    for section in ("root_causes", "soil_amendments", "precautions", "prevention"):
        if rec[section]:
            print(f"\n{section.replace('_', ' ').title()}:")
            for item in rec[section]:
                print(f"  - {item}")


if __name__ == "__main__":
    if len(sys.argv) < 2:
        sys.exit("Usage: python inference.py <image_path>")
    print_report(PlantDiagnoser().predict(sys.argv[1]))

In [ ]:
%%writefile make_dummy_data.py
"""Generate SYNTHETIC colour-blob images so you can smoke-test the whole pipeline
(train -> checkpoint -> app) before downloading a real dataset.
These images are NOT real plants; accuracy on them means nothing.

Usage: python make_dummy_data.py
"""
import random
from pathlib import Path

from PIL import Image, ImageDraw, ImageFilter

from config import DATA_DIR

BASE_COLOURS = {
    "healthy": (40, 140, 50),
    "nitrogen_deficiency": (190, 190, 70),
    "phosphorus_deficiency": (80, 60, 110),
    "potassium_deficiency": (60, 130, 50),     # green with brown edges (drawn below)
    "dead_necrotic": (90, 60, 35),
}

random.seed(0)
for name, base in BASE_COLOURS.items():
    out = DATA_DIR / name
    out.mkdir(parents=True, exist_ok=True)
    for i in range(24):
        jitter = tuple(max(0, min(255, c + random.randint(-20, 20))) for c in base)
        img = Image.new("RGB", (256, 256), (230, 225, 215))
        d = ImageDraw.Draw(img)
        d.ellipse((30, 30, 226, 226), fill=jitter)
        if name == "potassium_deficiency":
            d.ellipse((30, 30, 226, 226), outline=(120, 80, 30), width=18)
        if name == "dead_necrotic":
            for _ in range(15):
                x, y = random.randint(40, 200), random.randint(40, 200)
                d.ellipse((x, y, x + 30, y + 30), fill=(35, 25, 20))
        img = img.filter(ImageFilter.GaussianBlur(1.5))
        img.save(out / f"{name}_{i:03d}.jpg")
print(f"Wrote synthetic images to {DATA_DIR}")

## 2. Get your dataset into `/content/plant_health/data/`
Folder layout (one folder per class, name = label):
```
data/healthy/*.jpg
data/nitrogen_deficiency/*.jpg
data/phosphorus_deficiency/*.jpg
data/potassium_deficiency/*.jpg
data/dead_necrotic/*.jpg
```
Run **ONE** of the three options below (A, B, or C).

### Option A: upload a zip of your dataset
Zip so that the class folders are directly inside (or inside one top folder).

In [ ]:
from google.colab import files
import zipfile, shutil, os
up = files.upload()                      # choose your dataset .zip
zname = list(up.keys())[0]
shutil.rmtree("data_raw", ignore_errors=True)
zipfile.ZipFile(zname).extractall("data_raw")
# If the zip had a single top-level folder, step into it
root = "data_raw"
items = [d for d in os.listdir(root) if not d.startswith(("__MACOSX","."))]
if len(items) == 1 and os.path.isdir(os.path.join(root, items[0])):
    root = os.path.join(root, items[0])
shutil.rmtree("data", ignore_errors=True)
shutil.move(root, "data")
print("Classes found:", sorted(os.listdir("data")))

### Option B: dataset already on Google Drive
Edit `DRIVE_DATA_PATH` to the folder that contains the class folders.

In [ ]:
from google.colab import drive
import shutil
drive.mount('/content/drive')
DRIVE_DATA_PATH = "/content/drive/MyDrive/plant_data"     # <-- change this
shutil.rmtree("data", ignore_errors=True)
shutil.copytree(DRIVE_DATA_PATH, "data")
import os; print("Classes found:", sorted(os.listdir("data")))

### Option C: synthetic test data (only to check the pipeline works; the model will be meaningless)

In [ ]:
!python make_dummy_data.py

## 3. Check the dataset

In [ ]:
from pathlib import Path
exts = {".jpg",".jpeg",".png",".bmp",".webp"}
for d in sorted(Path("data").iterdir()):
    if d.is_dir():
        n = sum(1 for f in d.rglob("*") if f.suffix.lower() in exts)
        print(f"{d.name:<28} {n} images")

## 4. Train
Adjust `--epochs` if you like (15-25 is typical).

In [ ]:
!python train.py --data-dir data --epochs 15 --batch-size 32

## 5. Training curves

In [ ]:
import json, matplotlib.pyplot as plt
h = json.load(open("models/history.json"))
ep = [r["epoch"] for r in h]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(ep, [r["train_loss"] for r in h], label="train"); ax[0].plot(ep, [r["val_loss"] for r in h], label="val"); ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(ep, [r["train_acc"] for r in h], label="train"); ax[1].plot(ep, [r["val_acc"] for r in h], label="val"); ax[1].set_title("Accuracy"); ax[1].legend()
plt.show()

## 6. Quick test on one image (optional)

In [ ]:
from google.colab import files
from inference import PlantDiagnoser, print_report
up = files.upload()
d = PlantDiagnoser()
print_report(d.predict(list(up.keys())[0]))

## 7. Download the trained model
Put the downloaded file into your VS Code project at `models/plant_health_effnet_b0.pt`.

In [ ]:
from google.colab import files
files.download("models/plant_health_effnet_b0.pt")

In [ ]:
# Optional: also keep a copy on Google Drive
from google.colab import drive
import shutil, os
drive.mount('/content/drive')
os.makedirs("/content/drive/MyDrive/plant_health_models", exist_ok=True)
shutil.copy("models/plant_health_effnet_b0.pt", "/content/drive/MyDrive/plant_health_models/")
print("Saved to Drive")